# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 clinical oncology dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the clinical dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Note: dataset.metadata is an object, not a dict; access via attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their respective `@id`s.

In [ ]:
# Examine all record sets and their corresponding fields (by @id)
record_sets = list(dataset.record_sets)
print(f"Available record sets ({len(record_sets)}):")
for rs in record_sets:
    print(f"  RecordSet name: {rs.name}  (@id: {rs.id})")
    print("    Fields:")
    for f in rs.fields:
        print(f"      - {f.name}: {f.id}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use only entity `@id`s for programmatic handling.

In [ ]:
# Build a list of record set ids for extraction
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs in record_sets:
    rid = rs.id
    df = pd.DataFrame(dataset.records(record_set=rid))
    dataframes[rid] = df
    print(f"Loaded {len(df)} records for record set '{rs.name}' (@id: {rid})")
    print(f"Sample columns: {df.columns.tolist()[:6]}")
    print()

# Identify the principal clinical table for analysis (commonly the largest DataFrame):
main_rs_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
main_df = dataframes[main_rs_id]
print(f"Main record set: {main_rs_id}, shape: {main_df.shape}")
print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on clinical or pathological criteria, normalizing numeric fields, and grouping records by categorical variables for further analysis.

In [ ]:
# Display numeric-type fields (columns) with their @ids
numeric_fields = []
for rs in record_sets:
    if rs.id == main_rs_id:
        for f in rs.fields:
            if (hasattr(f, "data_type") and str(f.data_type).lower() in ["integer", "float", "number"]):
                numeric_fields.append(f.id)
if not numeric_fields:
    print("No numeric fields detected by schema; using all DataFrame numeric columns...")
    numeric_fields = main_df.select_dtypes(include=['number']).columns.tolist()
print("Numeric fields (@id):", numeric_fields)

if numeric_fields:
    numeric_field_id = numeric_fields[0]
    # If the columns are dicts (nested records), flatten or pick a simple numeric column
    if isinstance(main_df[numeric_field_id].iloc[0], dict):
        # Choose a subfield if appropriate (for demonstration)
        numeric_field = next(iter(main_df[numeric_field_id].iloc[0]))
        print(f"WARNING: {numeric_field_id} appears nested, selecting subfield: {numeric_field}")
        # Extract the subfield
        main_df[numeric_field_id+"_val"] = main_df[numeric_field_id].apply(lambda x: x.get(numeric_field))
        numeric_field_used = numeric_field_id+"_val"
    else:
        numeric_field_used = numeric_field_id

    threshold = main_df[numeric_field_used].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_used]) else 10
    filtered_df = main_df[main_df[numeric_field_used] > threshold]
    print(f"Filtered records where {numeric_field_used} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize selected numeric field
    filtered_df[f"{numeric_field_used}_normalized"] = (filtered_df[numeric_field_used] - filtered_df[numeric_field_used].mean()) / filtered_df[numeric_field_used].std()
    print(f"Normalized {numeric_field_used} for filtered records:")
    print(filtered_df[[numeric_field_used, f"{numeric_field_used}_normalized"]].head())
else:
    print("No numeric fields available for EDA.")

# Select a groupable categorical field (e.g. sex, anatomical site)
group_field_id = None
for rs in record_sets:
    if rs.id == main_rs_id:
        for f in rs.fields:
            if hasattr(f, 'data_type') and str(f.data_type).lower() == 'text':
                if f.id in main_df.columns and main_df[f.id].nunique() < len(main_df)/2:
                    group_field_id = f.id
                    break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_used].mean().to_frame(name=f"{numeric_field_used}_mean")
    print(f"Grouped by {group_field_id}, mean of {numeric_field_used}:")
    print(grouped_df.head())
else:
    print("No suitable group field detected.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Example: plot a histogram of the primary numeric variable, and a bar plot of group-wise means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields and numeric_field_used in main_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[numeric_field_used].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_used}")
    plt.xlabel(numeric_field_used)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10,5))
        sns.barplot(
            x=group_field_id,
            y=numeric_field_used,
            data=filtered_df,
            ci=None
        )
        plt.ylabel(f"Mean {numeric_field_used}")
        plt.title(f"Mean {numeric_field_used} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and examine a FAIR-compliant clinical dataset using the `mlcroissant` library, referencing all data entities strictly by their `@id`s. We:
- Loaded metadata and listed available record sets.
- Identified and displayed sample records and fields by `@id`.
- Extracted, filtered, normalized, and grouped data using DataFrames.
- Produced quick visualizations for numeric field distributions and group-wise analysis.
This workflow provides a reproducible and standards-driven approach for preliminary data exploration, ready for customized clinical or statistical analyses.